# Docs 2 — Getting the Text Out

A PDF is drawing instructions with text trapped inside. Today you build one
from raw bytes — about 700 of them — so the format stops being magic, then
extract your text back out and inspect what survived.

In [ ]:
# Build a real PDF by hand. Every PDF has these parts:
# a header, numbered objects (catalog -> pages -> page -> content -> font),
# a cross-reference table of byte offsets, and a trailer.
lines = [
    "COMMUNITY FOOD PANTRY - MONTHLY REPORT",
    "Date: March 14, 2026",
    "Families served: 212",
    "Donations received: $1,847.50",
    "Contact: pantry@example.org",
]
content = "BT /F1 12 Tf 72 720 Td 16 TL\n"
for ln in lines:
    safe = ln.replace("(", "\\(").replace(")", "\\)")
    content += f"({safe}) Tj T*\n"
content += "ET"
cbytes = content.encode()

objs = [
    b"<< /Type /Catalog /Pages 2 0 R >>",
    b"<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
    (b"<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 792] "
     b"/Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>"),
    b"<< /Length " + str(len(cbytes)).encode() + b" >>\nstream\n" + cbytes + b"\nendstream",
    b"<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>",
]

pdf = b"%PDF-1.4\n"
offsets = []
for i, o in enumerate(objs, 1):
    offsets.append(len(pdf))
    pdf += f"{i} 0 obj\n".encode() + o + b"\nendobj\n"
xref_pos = len(pdf)
pdf += f"xref\n0 {len(objs)+1}\n".encode() + b"0000000000 65535 f \n"
for off in offsets:
    pdf += f"{off:010d} 00000 n \n".encode()
pdf += (f"trailer\n<< /Size {len(objs)+1} /Root 1 0 R >>\n"
        f"startxref\n{xref_pos}\n%%EOF").encode()

with open("pantry_report.pdf", "wb") as f:
    f.write(pdf)
print(f"wrote pantry_report.pdf: {len(pdf)} bytes")
print("Open the Colab file browser (folder icon, left) and download it -")
print("it opens in any PDF viewer. You just wrote a file format by hand.")

## Now extract the text back out

`pypdf` reads the drawing instructions and rebuilds lines from character
positions. On our tidy hand-built file it works perfectly; on real-world
PDFs, this is exactly where the classic damage happens.

In [ ]:
%pip install -q pypdf
from pypdf import PdfReader

reader = PdfReader("pantry_report.pdf")
text = reader.pages[0].extract_text()
print(text)

## The inspection checklist

Run this on EVERY extraction, forever. Two minutes here saves hours
downstream.

In [ ]:
def inspect(text):
    print("INSPECTION")
    print(f"  characters: {len(text)},  lines: {len(text.splitlines())}")
    joined = [w for w in text.split() if sum(c.isdigit() for c in w) and sum(c.isalpha() for c in w) > 2]
    print(f"  suspicious digit/letter mixes (OCR or joined words): {joined[:5] or 'none'}")
    headers = [l for l in text.splitlines() if "Page " in l and " of " in l]
    print(f"  header/footer lines that leaked in: {headers or 'none'}")
    runs = text.count("  ")
    print(f"  double-space runs: {runs}")
    print("  -> now READ 10 lines yourself. The checklist finds patterns; you find surprises.")

inspect(text)

## Your turn

Upload any PDF you can legally use (Colab file browser → upload), extract
it, and run the checklist plus your own eyes on it.

## Turn-in

The inspection report for two real documents: what survived, what got
damaged, which classic casualties you found. "Nothing was damaged" is a
suspicious finding — say how you checked.